# Practical RSA Cryptanalysis & Vulnerability Exploitations
**Student:** Mahfuzur Rahman | **ID:** IT-23005  
**Course:** Cryptography and Cyber Law  
**Reference:** [IIT Bombay MNP Club - Number Theory & Cryptography Workshop](https://github.com/mnp-club/Number_Theory_and_Cryptography_Workshop)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IT23005/Cryptography-and-Cyber-Law/blob/main/CT-1%20Practicing%20Number%20Theory/03_RSA_Attacks_and_Cryptanalysis.ipynb)

---
## Overview
This notebook presents complete solutions and rigorous analysis for all **7 RSA Assignment Questions**:
- **Question 1:** Implementing RSA Encryption
- **Question 2:** Implementing RSA Decryption
- **Question 3:** Known-Plaintext / Dictionary Attack on Small Message Space (`DecipherSimple`)
- **Question 4:** Factorization Attack when $p < 1,000,000$ (`DecipherSmallPrime`)
- **Question 5:** Fermat's Factorization Attack when $|p - q| < 5000$ (`DecipherSmallDiff`)
- **Question 6:** Shared Prime Factor Attack via GCD (`DecipherCommonDivisor`)
- **Question 7:** Håstad's Broadcast Attack using Chinese Remainder Theorem ($e = 2$) (`DecipherHastad`)


### Core Helpers
String encoding, decoding, iterative modular exponentiation, and Extended Euclidean inversion.


In [ ]:
def ConvertToInt(s):
    res = 0
    for ch in s: res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = []
    while n > 0:
        res.append(chr(n % 256))
        n //= 256
    return "".join(reversed(res))

def PowMod(a, n, mod):
    res = 1
    base = a % mod
    exp = n
    while exp > 0:
        if exp % 2 == 1:
            res = (res * base) % mod
        base = (base * base) % mod
        exp //= 2
    return res

def ExtendedEuclid(a, b):
    if b == 0: return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0: b = (b % n + n) % n
    return b

def IntSqrt(n):
    if n <= 0: return 0
    low, high, ans = 1, n, 1
    while low <= high:
        mid = (low + high) // 2
        if mid * mid <= n:
            ans = mid
            low = mid + 1
        else:
            high = mid - 1
    return ans


### Question 1: RSA Encryption
Computes ciphertext $C \equiv M^e \pmod N$.


In [ ]:
def Encrypt(message, modulo, exponent):
    m_int = ConvertToInt(message)
    return PowMod(m_int, exponent, modulo)

print("Encrypted 'attack' with e=23917:", Encrypt("attack", 1000000007 * 1000000009, 23917))


### Question 2: RSA Decryption
Recovers $M \equiv C^d \pmod N$ using private key components $(p, q, e)$.


In [ ]:
def Decrypt(ciphertext, p, q, exponent):
    phiN = (p - 1) * (q - 1)
    d = InvertModulo(exponent, phiN)
    modulo = p * q
    m_int = PowMod(ciphertext, d, modulo)
    return ConvertToStr(m_int)

p = 1000000007
q = 1000000009
exponent = 23917
c = Encrypt("attack", p * q, exponent)
recovered = Decrypt(c, p, q, exponent)
print("Decrypted message:", recovered)
assert recovered == "attack" 


### Question 3: Attack 1 - Known-Plaintext / Dictionary Attack (`DecipherSimple`)
When messages come from a small candidate pool (e.g., Alice sending military orders), an attacker encrypts each possibility.


In [ ]:
def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for candidate in potential_messages:
        if Encrypt(candidate, modulo, exponent) == ciphertext:
            return candidate
    return "don't know"

modulo = 1000000007 * 1000000009
exponent = 65537
c = Encrypt("attack", modulo, exponent)
detected = DecipherSimple(c, modulo, exponent, ["attack", "don't attack", "wait"])
print("Deciphered message from dictionary:", detected)
assert detected == "attack" 


### Question 4: Attack 2 - Small Prime Factorization (`DecipherSmallPrime`)
If one prime factor is small ($p < 1,000,000$), trial division breaks the modulus instantly.


In [ ]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    found_p = None
    if modulo % 2 == 0:
        found_p = 2
    else:
        for i in range(3, 1000000, 2):
            if modulo % i == 0:
                found_p = i
                break
    assert found_p is not None, "Prime not found under 1,000,000"
    q = modulo // found_p
    return Decrypt(ciphertext, found_p, q, exponent)

# Test with p < 1,000,000
p4 = 999983
q4 = 10000000000000000051
n4 = p4 * q4
e4 = 65537
c4 = Encrypt("attack", n4, e4)
print("Decrypted small prime ciphertext:", DecipherSmallPrime(c4, n4, e4))


### Question 5: Attack 3 - Fermat's Factorization for Close Primes (`DecipherSmallDiff`)
When $|p - q| < 5000$, $N = a^2 - b^2$ where $a pprox \lceil \sqrt{N} ceil$ and $b = |p - q| / 2$.


In [ ]:
def DecipherSmallDiff(ciphertext, modulo, exponent):
    a = IntSqrt(modulo)
    if a * a < modulo:
        a += 1
    for _ in range(50000):
        t2 = a * a - modulo
        b = IntSqrt(t2)
        if b * b == t2:
            p = a - b
            q = a + b
            return Decrypt(ciphertext, p, q, exponent)
        a += 1
    raise ValueError("Factors not found")

p5 = 1000000007
q5 = 1000000009
n5 = p5 * q5
e5 = 239
c5 = Encrypt("attack", n5, e5)
print("Decrypted small diff ciphertext:", DecipherSmallDiff(c5, n5, e5))


### Question 6: Attack 4 - Shared Prime Factor Attack (`DecipherCommonDivisor`)
If two keys share a prime factor $p = \gcd(N_1, N_2)$, Euclid's algorithm factors both in $O(\log N)$.


In [ ]:
def GCD(a, b):
    while b: a, b = b, a % b
    return a

def DecipherCommonDivisor(c1, n1, e1, c2, n2, e2):
    p = GCD(n1, n2)
    assert p > 1
    q1 = n1 // p
    q2 = n2 // p
    return (Decrypt(c1, p, q1, e1), Decrypt(c2, p, q2, e2))

p6 = 101
q1_6 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
q2_6 = 10000000000000000051

n1 = p6 * q1_6
n2 = p6 * q2_6
c1 = Encrypt("attack", n1, 239)
c2 = Encrypt("wait", n2, 17)
print("Decrypted shared divisor messages:", DecipherCommonDivisor(c1, n1, 239, c2, n2, 17))


### Question 7: Attack 5 - Håstad's Broadcast Attack (`DecipherHastad`)
When the same message $M$ is broadcast with $e=2$ to 2 recipients, Chinese Remainder Theorem reconstructs $M^2$ over the integers, which is inverted by computing $\lfloor \sqrt{M^2} floor$.


In [ ]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    (x, y) = ExtendedEuclid(n1, n2)
    N = n1 * n2
    r = (r1 * n2 * y + r2 * n1 * x) % N
    return (r + N) % N

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):
    m2 = ChineseRemainderTheorem(first_modulo, first_ciphertext, second_modulo, second_ciphertext)
    m = IntSqrt(m2)
    assert m * m == m2, "Not an exact square"
    return ConvertToStr(m)

p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509
p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503
n1 = p1 * q1
n2 = p2 * q2

c1 = Encrypt("attack", n1, 2)
c2 = Encrypt("attack", n2, 2)
print("Hastad attack recovered message:", DecipherHastad(c1, n1, c2, n2))
